## DSPy Quantized Qwen2 Information Extraction

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast
import torch
import gc

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch

from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2','rougeL'], use_stemmer=True)
import json

#### Helper Functions

In [43]:
def validate_ans(example, pred, trace = None):

    gold = re.sub(r'\n|\s+ ', '',dict(example)['answer']).lower()
    print(gold)
    torch.cuda.empty_cache()
    gc.collect()
    try:
        prediction = str(json.loads(pred.answer.split('Answer: ')[1].lower()))
    except AttributeError: 
        prediction = pred.split('answer: ')[-1].lower()
    except: 
        prediction = str(pred.answer.lower())

    torch.cuda.empty_cache()
    gc.collect()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Test examples

In [3]:
test_examples = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\Manual Labeling - Sheet2.csv", header = None)

#### Set LLM Qwen2

In [4]:
llm = dspy.HFModel(model="unsloth/Qwen2-7B-bnb-4bit", hf_device_map='auto', model_kwargs= {'temperature':0.0,'do_sample': False, })
dspy.settings.configure(lm = llm)

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file m

#### Create DSPy Signature and Module

In [5]:
class GenerateAnswer(dspy.Signature):
    """Extract information from a job posting and return the output in a json format if you don't know answer Not Specified. Should be key-value with output as dictionary."""

    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(desc="key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words")

class QUESTIONANSWER(dspy.Module):
    def __init__(self,question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question=question

    def forward(self, context):
        context = context.replace('\n', ' ').replace('“', '"').replace('”', '"')
        context = normalize(context)
        question=self.question
        pred = self.generate_answer(context=context, question=question)
        torch.cuda.empty_cache()
        gc.collect()
        pred = re.sub(r"```\n|```", "",pred.answer)
        return dspy.Prediction(answer=pred)

In [6]:
uncompiled_fs=QUESTIONANSWER('''
                            "position_title" : What is the title of this position?
                            "location" : Where is this position located, including city, state and zip code?
                            "work_arrange" : What is the work arrangement for this position, remote, hybrid, or on-site?
                            "experience" : what are years of experience required for this position?
                            "employment_type" : What is the employment type, full time, part time, or internship?
                            "pay" : What is the pay for this position?
                            "degree" : What is required degree?
                            "certifications" : What certifications or qualifications are required?
                            "required_skills" : What are required skills?
                              ''')

#### Test Uncompiled DSPy 

In [7]:
print(test_examples[1][2])

NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility 
Inc. compensation: HOURLY - EVERY OTHER WEEKEND REQUIRED employment type: full-time job title: NURSE URSES - RNs or
LPNs Sudbury Pines Extended Care Sudbury, MA 01776 Sudbury Pines Extended Care Facility is seeking Nurses to join 
our team! Currently seeking the following positions: Full time or Part Time RN or LPN Sudbury Pines Extended Care 
facility is a 92 bed facility located in the MetroWest area. We are a single family owned facility who strives to 
provide quality care in a home-like and family oriented environment for our residents. Responsibilities: 
Responsible for the overall nursing care and delivery of resident services - medication pass, treatments, resident 
quality of life, etc. Manage staff and promote staff morale to ensure residents needs are being met in a proactive 
manner. Qualifications: Must have a valid MA Nursing License. Minimum of 1 year Long term care experience/SNF 
experience preferred - although new graduates welcome. We offer great benefits for all Full Time staff -some 
limitations apply for part time staff. Child Day Care facility on site since 1986 - infants through preschoolers - 
prorated for staff. Job Type: Full-time or part-time Job Types: Full-time, Part-time Benefits: 401(k) 401(k) 
matching Dental insurance Flexible schedule Health insurance Life insurance Paid time off Referral program Tuition 
reimbursement Physical setting: JCAHO accredited facility Long term care Nursing home Standard shift: Day shift 
Evening shift Night shift Supplemental schedule: Holidays Overtime Weekly schedule: Rotating weekends COVID-19 
considerations: All staff are required to follow current COVID 19 protocols as defined by the Commonwealth of MA - 
must be prepared to wear masks, and follow other infection control protocols as well as all vaccination guidelines 
expected to be employed in a LTC SNF Ability to commute/relocate: Sudbury, MA 01776: Reliably commute or planning 
to relocate before starting work (Required) Experience: Nursing to: 1 year (Preferred) License/Certification: RN or
LPN License in Massachusetts (Required) Work Location: One location Principals only. Recruiters, please don't 
contact this job poster. Do NOT contact this job poster with unsolicited services or offers. post id: 7694393986 
updated: [ ]

In [8]:
with dspy.context(lm = llm):
    pred = uncompiled_fs(context = test_examples[1][2])
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:693: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{
    'position_title': 'Nurses - RNS or LPNs',
    'location': 'Sudbury, MA 01776',
    'work_arrange': 'On-site',
    'experience': 'Minimum of 1 year long term care experience/SNF experience preferred',
    'employment_type': 'Full-time or Part-time',
    'pay': 'Hourly',
    'degree': 'Valid MA Nursing License',
    'certifications': 'RN or LPN license in Massachusetts',
    'required_skills': 'Nursing'
}

#### Create Train, Test Format

In [9]:
train_example_list = [
    """
    {
        "position_title": "Derrickhand",
        "location": "Buckhannon, West Virginia 26201",
        "work_arrangement": "On-Site, Shifts",
        "experience": "1-2 years of Derrickhand experience",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "High school diploma/GED or equivalent",
        "certification": "CDL B License",
        "required_skills": "Effective verbal/written communication in English, ability to interact with teams in a fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent customer-service"
    }
    """,
    """
    {
        "position_title": "Pharmacy Technician",
        "location": "San Quentin, California",
        "work_arrangement": "On-Site, Shifts, Relocation required if applicable",
        "experience": "1 year of experience as Pharmacy Technician",
        "employment_type": "Contract",
        "pay": "$18-$19/hr",
        "degree": "High school diploma or GED",
        "certification": "Pharmacy Technician Certification, BLS Certification",
        "required_skills": "Excellent communication skills, ability to use computer for day-to-day tasks, basic math for counting medications"
    }
    """,
    """
    {
        "position_title": "GIS Technician",
        "location": "Oklahoma City, OK 73134",
        "work_arrangement": "On-Site, Shifts, Relocation required if applicable",
        "experience": "3-5 years of GIS experience",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "Bachelor's degree in a related field",
        "certification": "Not specified",
        "required_skills": "GIS, ArcPy, ESRI ArcGIS Desktop or ArcPro, Field Maps/ArcGIS Online, Microsoft Office suites, clerical skills, ability to work in a team environment, initiative in recognizing need for improvements of existing systems, tracking down msising/misfiled items, filing accuracy"
    }
    """,
    """
    {
        "position_title": "Graphic Designer",
        "location": "Goochland, VA",
        "work_arrangement": "Hybrid, with two in-office days per week",
        "experience": "Minimum 5 years design and publications experience",
        "employment_type": "Part-time",
        "pay": "Not specified",
        "degree": "College degree in graphic design, visual arts, or related field",
        "certification": "Not specified",
        "required_skills": "proficiency with InDesign, Photoshop, Illustrator, working knowledge of Constant Contact, strong organizational skills, excellent oral/written communication and client-relations skills, ability to work under pressure, working knowledge of AP style, 35mm and digital photography skills, Mac environment"
    }
    """,
    """
    {
        "position_title": "Senior Cybersecurity Analyst",
        "location": "Washington DC, USA",
        "work_arrangement": "Not Specified",
        "experience": "5 years of experience in cybersecurity",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "Bachelor's degree",
        "certification": "DOD 8570 Level II, DOD 8570 Level III or Manager",
        "required_skills": "carbon black implementation, splunk, CDM dashboards, CI/CD, black box testing of IT assets"
    }
    """,
    """
    {
        "position_title": "Academic Instructor",
        "location": "Fullerton, CA",
        "work_arrangement": "Not specified",
        "experience": "Experience working with low-income and diverse student population, experience working with AUHSD student, experience working with middle or high school students",
        "employment_type": "Part-time",
        "pay": "$47-$52/hr",
        "degree": "Bachelor's degree, Master's degree",
        "certification": "Not specified",
        "required_skills": "Teaching, ability to work in a collaborative team environment, develop effective teaching strategies, lifting of up to 25lbs"
    }
    """,
    """
    {
        "position_title": "Retail Scan Associate",
        "location": "Luverne, Minnesota",
        "work_arrangement": "On-site",
        "experience": "Not specified",
        "employment_type": "Part-time",
        "pay": "$16/hr",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Ability to endure being on your feet for long periods of time, ability to lift up to 25lbs, reach 6 feet in the air, ability to perform repetitive movements with hands, wrists, arms, and legs, attention to detail and ability to work independently"
    }
    """,
    """
    {
        "position_title": "Machinist Operator",
        "location": "Valencia, CA",
        "work_arrangement": "On-site, Shifts",
        "experience": "3 years of CNC operating and programming experience, experience with BobCad",
        "employment_type": "Full-time, Temp-to-Hire",
        "pay": "$30/hr",
        "degree": "High school diploma/GED",
        "certification": "BobCad, Solid Works, HAAS ST 20, Fadal WMC4020, Akira-Seiki SL20",
        "required_skills": "Strong math, problem solving and analytical skills"
    }
    """,
    """
    {
        "position_title": "Automotive Technician",
        "location": "Bremerton, WA",
        "work_arrangement": "On-site",
        "experience": "4-7 years of experience",
        "employment_type": "Full-time",
        "pay": "$50,000 - $83,200 a year",
        "degree": "High school diploma",
        "certification": "Not specified",
        "required_skills": "capable of diagnosing and repairing any system of the automobile to dealership and manufacturer's standards without supervision"
    }
    """,
    """
    {
        "position_title": "Industrial Maintenance Electrician",
        "location": "Cary, NC",
        "work_arrangement": "On-site, Shifts",
        "experience": "previous experience working in a food manufacturing plant",
        "employment_type": "Full-time",
        "pay": "$30.53/hr",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Basic computer skills including Microsoft Office, Demonstrated knowledge of behavior-based safety systems, ability to lift up to 50lbs"
    }
    """,
    """
    {
        "position_title": "Technician",
        "location": "Palm Harbor, FL",
        "work_arrangement": "On-site",
        "experience": "5+ years of service technician experience, 10+ preferred",
        "employment_type": "Full-time",
        "pay": "$27K - $66K",
        "degree": "High school diploma/GED",
        "certification": "ASE Certification, Diagnostic, Electric and Engine Repair",
        "required_skills": "excellent hand-eye coordination, mechanical and troubleshooting skills, ability to operate electronic diagnostic equipment, excellent customer service skills, basic computer competencies, ability to collaborate with others, ability to learn new technology"
    }
    """,
    """
    {
        "position_title": "Director of Email Marketing",
        "location": "Austin, TX",
        "work_arrangement": "Not specified",
        "experience": "5+ years of experience as project manager",
        "employment_type": "Full-time",
        "pay": "$125,000 - $250,000 a year",
        "degree": "Bachelor's degree in marketing",
        "certification": "Not specified",
        "required_skills": "Ability to collaborate with a team of people to learn and grow and own new responsibilities, sophisticated verbal and written communication, people, and leadership skills, advanced analytical and problem-solving skills, focus on email marketing, experience with testing and activating cold audiences, experience with project methodologies including agile, waterfall, and scrum, experience having worked at a startup or a small company with less than 50 people"
    }
    """,
    """
    {
        "position_title": "Campus Store Leader",
        "location": "Philadelphia, PA",
        "work_arrangement": "On-site",
        "experience": "0-5 years of relevant experience, retail experience is a plus",
        "employment_type": "Full-time",
        "pay": "$15.00 - $21.56 an hour",
        "degree": "Associate's degree",
        "certification": "Not specified",
        "required_skills": "analysis skills, computer skills, financial acumen, communication skills, time management, advanced relationship building, ability to influence a team, customer outreach"
    }
    """,
    """
    {
        "position_title": "Dental Laboratory Technician",
        "location": "Wood Dale, IL",
        "work_arrangement": "On-site",
        "experience": "5 years of Fabrication experience",
        "employment_type": "Part-time, Contract",
        "pay": "$14.00 - $22.00 per hour",
        "degree": "High school diploma or equivalent",
        "certification": "Not specified",
        "required_skills": "3Shape software savvyy, Fabrication of custom trays, bite rims, denture/partial repairs"
    }
    """,
   

 """
    {
        "position_title": "Senior Data Engineer (Machine Learning)",
        "location": "Wood Dale, IL",
        "work_arrangement": "Remote",
        "experience": "5+ years of experience as a Data Engineer",
        "employment_type": "Full-time, Direct Hire",
        "pay": "$130,000 - $170,000 per year",
        "degree": "Bachelor's degree in computer science, engineering, or data science",
        "certification": "Not specified",
        "required_skills": "focused on Python, Postgres, and DBT, Experience using Google Cloud Platform and ideally Google AI tools, Experience with data modeling, data warehousing, and building ETL pipelines with DBT, Python, Postgres, DBT, SQL, PostgreML, TensorFlow, PyTorch, Google BigQuery, Airflow, Fivetran, Make, excellent understanding of machine learning algorithms, processes, tools and platforms, proven ability to drive business results with data-based insights"
    }
    """,
    """
    {
        "position_title": "Solutions Architect",
        "location": "Mayfield Heights, OH",
        "work_arrangement": "Not Specified",
        "experience": "Minimum 2 years of experience",
        "employment_type": "Full-time",
        "pay": "$64.7K - $81.9K a year",
        "degree": "Bachelors Degree in Information Technology, MIS, or Financial Accounting, Advanced diploma/degree in Finance/Management Accounting preferred",
        "certification": "CPA Certification preferred, Certification in SAP FICO highly preferred",
        "required_skills": "supporting SAP FI/CO modules and related integration with MM and SD SAP AP, AR, COPA and COPC, Minimum of 2 full SAP life-cycle implementations, and upgrades, Experience with SAP S/4 HANA Upgrade a plus, SAP FI/CO modules, MM, SD, SAP, AP, AR, COPA, COPC"
    }
    """,
    """
    {
        "position_title": "Marketing Manager",
        "location": "Andover, NJ 07821",
        "work_arrangement": "Remote",
        "experience": "Minimum of 2 years in marketing & social media management preferred, demonstrable experience with social analytics tools",
        "employment_type": "Full-time, Shift and schedule weekends as needed, nights as needed",
        "pay": "$60,000 a year",
        "degree": "Bachelor's degree in marketing, communications, or related field",
        "certification": "Not specified",
        "required_skills": "excellent writing, editing(photo/video/text), presentation, and communication skills, ability to work nights and weekends as needed"
    }
    """,
    """
    {
        "position_title": "Technical Project Manager",
        "location": "Santa Monica, CA",
        "work_arrangement": "On-site",
        "experience": "4-7 years of experience in project management",
        "employment_type": "Full-time",
        "pay": "$78,000 - $150,000",
        "degree": "Bachelor's degree in a hardware or software engineering field",
        "certification": "PMP or similar certification",
        "required_skills": "Ability to effectively communicate project milestones, status, and risks at all levels of the organization, delivering consumer, enterprise, or industrial products, experience managing all steps of the product lifecycle from concept to manufacturing, experience simultaneously managing different projects with multiple stakeholders, demonstrated experience developing or deploying applications of interactive, AR, VR, or XR, experience as a scrum master or similar agile processes, experience interfacing with test teams, experience working on defense & aerospace products"
    }
    """,
    """
    {
        "position_title": "Logistics Coordinator",
        "location": "Henderson, NV 89074",
        "work_arrangement": "On-site",
        "experience": "previous experience in operations or a related field is a plus",
        "employment_type": "Full-time, Shift",
        "pay": "From $14 an hour",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Excellent organizational and inter-personal and communication skills, strong organizational and multitasking abilities, proficiency in Microsoft Office and other relevant software"
    }
    """,
    """
    {
        "position_title": "Sr. Net Developer with AWS",
        "location": "Irving, TX",
        "work_arrangement": "On-site",
        "experience": "Experience with microservices, cloud services, especially with AWS, 2+ years of experience",
        "employment_type": "Full-time, Contract 12+ Months",
        "pay": "Not specified",
        "degree": "Bachelor's/Master's in computer science or related fields",
        "certification": "AWS certified(associate or professional)",
        "required_skills": "proficiency with C#, .NET Core framework 2.x & higher, Git, SVN, MongoDB, Cassandra, familiarity with dev-ops software development methods and Docker container related technologies"
    }
    """
]

In [10]:
test_examples_list = [
    '{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": "Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and price quotes to the DOD", "required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}',
    '{"position_title": "Penetration Tester", "location": "Washington, DC", "work_arrangement": "On-site", "experience": "10+ years of Penetration Testing experience", "employment_type": "Full time", "pay": "Not specified", "degree": "Bachelors Degree in Computer Science", "certification": "Offensive Security certification (OSCP, OSCE), GIAC certification (GPEN, GWAPT, GXPN), or technology specific certification (MCSE, LPIC, CCNA)", "required_skills": "NIST guidance, FedRAMP control baseline, industry best practice"}',
    '{"position_title": "NURSES - RNs or LPNs", "location": "Sudbury, MA 01776", "work_arrangement": "on-site", "experience": "Minimum of 1 year Long term care experience/SNF experience preferred", "employment_type": "full-time", "pay": "HOURLY - EVERY OTHER WEEKEND REQUIRED", "degree": "Must have a valid MA Nursing License", "certification": "RN or LPN License in Massachusetts", "required_skills": "Medication pass, treatments, resident care"}',
    '{"position_title": "Planner IV - Transportation Planner", "location": "Yakima, WA, 98901", "work_arrangement": "On-site", "experience": "5 years of increasingly responsible professional experience", "employment_type": "Full-Time", "pay": "$39.84 - $50.53 Hourly", "degree": "Bachelor\'s Degree in Planning or other related field", "certification": "None specified", "required_skills": "Transportation planning, coordination with the Yakama Nation, preparation of loans and grants"}',
    '{"position_title": "Associate Attorney", "location": "McAllen, TX", "work_arrangement": "on-site", "experience": "None specified.", "employment_type": "full-time", "pay": "$50,000 a year", "degree": "Law doctoral degree", "certification": "Admission to the state bar and in good standing with the relevant jurisdiction.", "required_skills": "Interest in Family and Criminal Law, proven track record of successful hearing coverage and strong advocacy skills."}',
    '{"position_title": "EMT-Advanced-Emergency Medical Service", "location": "Rosenberg, TX 77471", "work_arrangement": "On-site", "experience": "Pre-hospital experience preferred, experience in a high performance ALS system", "employment_type": "Full time", "pay": "$2,008.28 - $2,421.41 biweekly", "degree": "High school diploma/GED", "certification": "paramedic certification or EMS degree, AEMT, enrolled in an EMT Paramedic Program, DSHS EMT-Advanced, valid Texas driver\'s license", "required_skills": "Strong verbal and written communication, organizational skills, interpersonal skills, judgment, reasoning, decision-making, teaching"}',
    '{"position_title": "Transportation Environmental Resources Specialist", "location": "Weston, West Virginia 26452-8289", "work_arrangement": "On-site", "experience": "24 Months", "employment_type": "Full time Permanent", "pay": "$1,700.00 - $2,521.15 Biweekly", "degree": "Bachelor\'s degree from a regionally accredited college or university with a major in archeology, chemistry, geology, history, physics, geography, biology, economics, engineering, environmental studies, natural science, or a related field.", "certification": "Drivers license, DL", "required_skills": "Full-performance level, complex professional work in a specialty area in the acquisition, preservation, management and protection of the state\'s environmental/natural resources."}',
    '{"position_title": "Hotel Front Desk Clerk", "location": "La Quinta Inn & Suites, USF Tampa, FL", "work_arrangement": "On-site", "experience": "At least one year of hospitality industry experience", "employment_type": "Full Time", "pay": "$14 hourly", "degree": "High school diploma or GED", "certification": "None specified", "required_skills": "Customer service, Microsoft Office, organizational skills, communication, time management"}',
    '{"position_title": "Psychotherapist", "location": "Asbury, NJ", "work_arrangement": "On-site", "experience": "1 year", "employment_type": "Hourly", "pay": "$65 - $95 an hour", "degree": "Doctor of Psychology Doctoral degree or equivalent", "certification": "LSW Social Work License, LCSW, LPC, LAC, or other relevant licenses", "required_skills": "experience with children, strong interpersonal skills, ability to establish rapport with clients"}',
    '{"position_title": "Cryptocurrency / FX Trader - Entry Level", "location": "Not specified", "work_arrangement": "Remote", "experience": "No prior experience required", "employment_type": "Full-time or part-time", "pay": "Results-based commissions and performance bonuses", "degree": "Bachelor\'s degree in finance, economics, or related field preferred", "certification": "None specified", "required_skills": "Strong analytical skills, quick decision-making"}'
]

In [11]:
test_results = test_examples_list
test_contents = list(test_examples[1])
test_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(test_contents, test_results)]
testset = test_examples_list
testset = [x.with_inputs('context') for x in testset]

In [12]:
train_examples = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\Manual Labeling - Sheet1.csv")

In [13]:
train_results = train_example_list
train_contents = list(train_examples['body'])
train_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(train_contents, train_results)]
trainset=train_examples_list[:2]
trainset = [x.with_inputs('context') for x in trainset]

#### Test on Test set

In [14]:
answ = testset[0]
print(answ.answer)

{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": 
"Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A 
BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and price quotes to the DOD", 
"required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}

In [15]:
with dspy.context(lm=llm):
    pred = uncompiled_fs(context=testset[0].context)
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))


c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{
    'position_title': 'Federal Sales Engineer',
    'location': 'Walpole, MA',
    'work_arrange': 'Hybrid',
    'experience': 'Experience in the development and selling of power electronics or similar hardware, along with 
providing technical support, to the DOD, homeland security, prime contractors, and commercial industries (telecom, 
medical, robotics, energy, etc.)',
    'employment_type': 'Full Time',
    'pay': '$120k/year',
    'degree': 'BS in the engineering field',
    'certifications': 'Working knowledge of ERP (Epicor is preferred)',
    'required_skills': 'Inside/outside technical sales experience, experience working with outside sales reps, RFQ 
experience and price quotes to the DOD preferred, working knowledge of ERP (Epicor is preferred), familiar with 
power electronics and has sales experience in the military and industrial sectors, military service is considered a
plus'
}

In [16]:
validate_ans(answ, pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{'position_title': 'federal sales engineer', 'location': 'walpole, ma', 'work_arrange': 'hybrid', 'experience': 
'experience in the development and selling of power electronics or similar hardware, along with providing technical
support, to the dod, homeland security, prime contractors, and commercial industries (telecom, medical, robotics, 
energy, etc.)', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'bs in the engineering field', 
'certifications': 'working knowledge of erp (epicor is preferred)', 'required_skills': 'inside/outside technical 
sales experience, experience working with outside sales reps, rfq experience and price quotes to the dod preferred,
working knowledge of erp (epicor is preferred), familiar with power electronics and has sales experience in the 
military and industrial sectors, military service is considered a plus'}

0.3632122341251358

0.3632122341251358

In [17]:
teleprompter = BootstrapFewShot(metric=validate_ans)
compiled_info_extract = teleprompter.compile(uncompiled_fs, trainset=trainset)

  0%|          | 0/2 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

{ "position_title": "pharmacy technician", "location": "san quentin, california", "work_arrangement": "on-site, 
shifts, relocation required if applicable", "experience": "1 year of experience as pharmacy technician", 
"employment_type": "contract", "pay": "$18-$19/hr", "degree": "high school diploma or ged", "certification": 
"pharmacy technician certification, bls certification", "required_skills": "excellent communication skills, ability
to use computer for day-to-day tasks, basic math for counting medications" }

---

follow the following format.

context: contain relevant facts

question: unique possible questions

reasoning: let's think step by step in order to ${produce the answer}. we ...

answer: key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, 
certifications, required_skills each with less than 20 words

---

context: derrickhand, buckhannon, wv   job order number          wv2925359   post date          05/12/2023   job 
location          buckhannon, west virginia 26201   county          upshur   job summary          job summary: this
position is a crew member assigned to work on a well service rig, responsible for performing services on oil and 
gas wells. duties include performing all well-servicing tasks from an elevated position (rod basket or tubing 
board), assisting in rigging up or down, picking up or laying down tubing, and other functions specified by the 
customer or well operator. this position has a dotted line reporting line to: rig supervisor. responsibilities: 
assists the operator in rigging up and down, lining up the well service rig with the well. sets hydraulic jacks, 
handles pads/boards and assists in attaching the guy wires to the anchor. responsible for all elevated work 
associated with rigging up/down (i.e. removing horse head from pumping unit). responsible for all work performed 
for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the elevator), 
performs servicing on the well. drives the crew truck as needed. operates tubing elevators for standing tubing in 
derrick. assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the work floor
or vice versa. assists in walking the rods when laying down rods. reports any safety hazards, accidents or 
maintenance issues to the rig supervisor. ensures that work carried out is in compliance with company policies and 
procedures and according to safety regulations. may be required to work floors or operate the rig when needed. 
performs other related duties as assigned. preferred qualifications: 1-2 years of workover - derrickhand experience
required. ability to effectively communicate, both verbally and written. ability to interact with others in a team 
environment. ability to work in a fast-paced environment and handle multiple tasks at once. basic problem solving 
and organizational skills. excellent customer service skills, to provide world class value to customers cdl b 
license is required to drive rig. must meet all qualifications defined in the motor vehicle policy if required to 
drive. ability to communicate verbally and in writing, in english, is preferred. education requirements: high 
school diploma, ged, or the equivalent is preferred. we are proud to offer a very competitive compensation and 
benefits package including: medical insurance vision and dental insurance life insurance 401(k) education 
assistance short-term disability paid time-off request priority protected veteran referrals equal opportunity 
employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender identity   
experience          0 months   pay rate          0 $ / hour   master group          construction and extraction 
occupations   job type          rotary drill operators, oil and gas   shift          day shift

question: "position_title" : what is the title of this position? "location" : where is t

0.0619277210166343

 50%|█████     | 1/2 [00:56<00:56, 56.08s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{"position_title": "pharmacy technician","location": "san quentin, california","work_arrangement": "on-site, 
shifts, relocation required if applicable","experience": "1 year of experience as pharmacy 
technician","employment_type": "contract","pay": "$18-$19/hr","degree": "high school diploma or 
ged","certification": "pharmacy technician certification, bls certification","required_skills": "excellent 
communication skills, ability to use computer for day-to-day tasks, basic math for counting medications"}

{ "position_title": "derrickhand", "location": "buckhannon, west virginia 26201", "work_arrangement": "on-site, 
shifts", "experience": "1-2 years of derrickhand experience", "employment_type": "full-time", "pay": "not 
specified", "degree": "high school diploma/ged or equivalent", "certification": "cdl b license", "required_skills":
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service" }

---

follow the following format.

context: contain relevant facts

question: unique possible questions

reasoning: let's think step by step in order to ${produce the answer}. we ...

answer: key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, 
certifications, required_skills each with less than 20 words

---

context: pharmacy technician san quentin, ca $18 - $19 an hour - contract profile insights find out how your skills
align with the job description licenses do you have a valid pharmacy technician license license? yes no 
certifications do you have a valid pharmacy technician certification certification? yes no skills do you have 
experience in customer service ? yes no education do you have a high school diploma or ged ? yes no job details 
heres how the job details align with your . pay $18 - $19 an hour job type contract shift and schedule morning 
shift monday to friday location san quentin, ca full job description j ob: pharmacy technician required:-pharmacy 
technician license, basic life support, covid. note- please apply to the job, if you meet the required job 
criteria. location: san quentin, ca, 94964 facility pay: $18.00/hr to $19.00/hr on w2 duration: 28 weeks contract 
shift : mon to fri 7:15 am to 3:45 pm morning shift job description: pharmacy technician job description we are 
looking to hire a pharmacy technician to be responsible for handling customer transactions and inventory at our 
pharmacy. you will be dispensing both prescription and over the counter medicine to customers. you will also be 
responsible for checking and evaluating written prescriptions, as well as recording and verifying patient 
information while maintaining confidentiality and compliance with regulations. in order to be a successful 
candidate, you must display excellent communication skills and an affinity for great customer service. you must be 
comfortable with working in a fast-paced work environment. pharmacy technician responsibilities: accepting written 
prescriptions or refill requests from patients and evaluating information for completeness and accuracy. recording 
medical histories while maintaining confidentiality and compliance with hipaa regulations. delivering quality 
customer service to patients, responding to inquiries, questions, or requests, and referring them to the pharmacist
for medical information. verifying the accuracy of patient information. counting prescription medication, filling 
prescriptions, and typing and attaching medication labels. performing inventory audits and purchasing supplies and 
medication. process patient insurance. pharmacy technician requirements: a high school diploma or equivalent. a 
pharmacy technician certification. excellent communication skills. the ability to use a computer for day-to-day 
tasks. basic math skills for counting medications. customer service experience. experience in a fast paced work 
environment. #4007 job type: contract salary: $18.00 - $19.00 per hour schedule: monday to friday morning shift 
experience: pharmacy technician: 1 year (required) license/certification: pharmacy technician (required) bls 
certification (required) ability to relocate: san quentin, ca: relocate before starting work (required) work 
location: in person if you require alternative methods of application or screening, you must approach the employer 
directly to request this as indeed is not r

0.0675716597318539

100%|██████████| 2/2 [01:22<00:00, 41.32s/it]


In [91]:
with dspy.context(lm=llm):
    pred = compiled_info_extract(context=testset[0].context)
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))

In [45]:
validate_ans(answ, pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, 
certifications, required_skills each with less than 20 words

---

context: derrickhand, buckhannon, wv   job order number          wv2925359   post date          05/12/2023   job 
location          buckhannon, west virginia 26201   county          upshur   job summary          job summary: this
position is a crew member assigned to work on a well service rig, responsible for performing services on oil and 
gas wells. duties include performing all well-servicing tasks from an elevated position (rod basket or tubing 
board), assisting in rigging up or down, picking up or laying down tubing, and other functions specified by the 
customer or well operator. this position has a dotted line reporting line to: rig supervisor. responsibilities: 
assists the operator in rigging up and down, lining up the well service rig with the well. sets hydraulic jacks, 
handles pads/boards and assists in attaching the guy wires to the anchor. responsible for all elevated work 
associated with rigging up/down (i.e. removing horse head from pumping unit). responsible for all work performed 
for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the elevator), 
performs servicing on the well. drives the crew truck as needed. operates tubing elevators for standing tubing in 
derrick. assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the work floor
or vice versa. assists in walking the rods when laying down rods. reports any safety hazards, accidents or 
maintenance issues to the rig supervisor. ensures that work carried out is in compliance with company policies and 
procedures and according to safety regulations. may be required to work floors or operate the rig when needed. 
performs other related duties as assigned. preferred qualifications: 1-2 years of workover - derrickhand experience
required. ability to effectively communicate, both verbally and written. ability to interact with others in a team 
environment. ability to work in a fast-paced environment and handle multiple tasks at once. basic problem solving 
and organizational skills. excellent customer service skills, to provide world class value to customers cdl b 
license is required to drive rig. must meet all qualifications defined in the motor vehicle policy if required to 
drive. ability to communicate verbally and in writing, in english, is preferred. education requirements: high 
school diploma, ged, or the equivalent is preferred. we are proud to offer a very competitive compensation and 
benefits package including: medical insurance vision and dental insurance life insurance 401(k) education 
assistance short-term disability paid time-off request priority protected veteran referrals equal opportunity 
employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender identity   
experience          0 months   pay rate          0 $ / hour   master group          construction and extraction 
occupations   job type          rotary drill operators, oil and gas   shift          day shift

question: "position_title" : what is the title of this position? "location" : where is this position located, 
including city, state and zip code? "work_arrange" : what is the work arrangement for this position, remote, 
hybrid, or on-site? "experience" : what are years of experience required for this position? "employment_type" : 
what is the employment type, full time, part time, or internship? "pay" : what is the pay for this position? 
"degree" : what is required degree? "certifications" : what certifications or qualifications are required? 
"required_skills" : what are required skills?

reasoning: let's think step by step in order to extract information from a job posting and return the output in a 
json format if you don't know answer not specified. should be key-value with output as di

0.010007207316340425

0.010007207316340425

In [19]:
compiled_info_extract.save("compiled_evaluate_v2.json")

#### Evaluate Uncompiled Quantized Unsloth

In [61]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10)

prev_score=evaluation(uncompiled_fs, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{'position_title': 'federal sales engineer', 'location': 'walpole, ma', 'work_arrange': 'hybrid', 'experience': 
'experience in the development and selling of power electronics or similar hardware, along with providing technical
support, to the dod, homeland security, prime contractors, and commercial industries (telecom, medical, robotics, 
energy, etc.)', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'bs in the engineering field', 
'certifications': 'working knowledge of erp (epicor is preferred)', 'required_skills': 'inside/outside technical 
sales experience, experience working with outside sales reps, rfq experience and price quotes to the dod preferred,
working knowledge of erp (epicor is preferred), familiar with power electronics and has sales experience in the 
military and industrial sectors, military service is considered a plus'}

0.3632122341251358

...

{"position_title": "cryptocurrency / fx trader - entry level", "location": "not specified", "work_arrangement": 
"remote", "experience": "no prior experience required", "employment_type": "full-time or part-time", "pay": 
"results-based commissions and performance bonuses", "degree": "bachelor's degree in finance, economics, or related
field preferred", "certification": "none specified", "required_skills": "strong analytical skills, quick 
decision-making"}

{'position_title': 'un cryptocurrency / fx trader - entry level', 'location': 'remote (sugar 
land/pasadena/katy/spring/the woodlands/remote)', 'work_arrange': 'remote', 'experience': 'no prior experience is 
required', 'employment_type': 'contract', 'pay': 'results-based commissions and performance bonuses', 'degree': 
"bachelor's degree in finance, economics, or a related field is preferred but not required", 'certifications': 'no 
specific certifications are required', 'required_skills': 'strong motivation and drive to succeed as a trader, 
willingness to develop a strong understanding of financial markets and risk management, strong analytical skills 
and the ability to make quick decisions in a fast-paced environment, ability to work in a fast-paced and 
mentally-challenging environment'}

0.3842159916926272

Average Metric: 5.993252862344113 / 10  (59.9): 100%|██████████| 10/10 [04:26<00:00, 26.63s/it]


,example_context,example_answer,pred_context,pred_answer,validate_ans
0,"Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA Advanced Recruiting Solutions Walpole, MA Depends on Experience Full Time Work...","{""position_title"": ""Senior Inside Sales Rep/Sales Engineer"", ""location"": ""Walpole, MA"", ""work_arrangement"": ""Hybrid"", ""experience"": ""Depends on Experience"", ""employment_type"": ""Full Time"", ""pay"": ""$120K/year"", ""degree"": ""A BS in the...","federal sales engineer - tech & isr experience - hybrid remote - walpole ma advanced recruiting solutions walpole, ma depends on experience full time work...","key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: federal sales engineer - tech...",✔️ [0.3632122341251358]
1,"Position Description Penetration Tester Location Washington, DC Req # 12763 # of openings 2 ECS is seeking a Penetration Tester to work in our Washington,...","{""position_title"": ""Penetration Tester"", ""location"": ""Washington, DC"", ""work_arrangement"": ""On-site"", ""experience"": ""10+ years of Penetration Testing experience"", ""employment_type"": ""Full time"", ""pay"": ""Not specified"", ""degree"": ""Bachelors Degree in...","position description penetration tester location washington, dc req # 12763 # of openings 2 ecs is seeking a penetration tester to work in our washington,...","key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: position description penetration tester location...",✔️ [0.6100106923282544]
2,NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility Inc. compensation: HOURLY...,"{""position_title"": ""NURSES - RNs or LPNs"", ""location"": ""Sudbury, MA 01776"", ""work_arrangement"": ""on-site"", ""experience"": ""Minimum of 1 year Long term care experience/SNF experience preferred"", ""employment_type"": ""full-time"",...",nurses - rns & lpns in snf - sign on bonus - child daycare on site (sudbury) sudbury pines extended care facility inc. compensation: hourly...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: nurses - rns & lpns...",✔️ [0.9072653061224489]
3,Planner IV - Transportation Planner Job Details Apply Print Share This listing closes on 7/17/2023 at 11:59 PM Pacific Time (US & Canada); Tijuana. Salary...,"{""position_title"": ""Planner IV - Transportation Planner"", ""location"": ""Yakima, WA, 98901"", ""work_arrangement"": ""On-site"", ""experience"": ""5 years of increasingly responsible professional experience"", ""employment_type"": ""Full-Time"", ""pay"": ""$39.84 -...",planner iv - transportation planner job details apply print share this listing closes on 7/17/2023 at 11:59 pm pacific time (us & canada); tijuana. salary...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: planner iv - transportation planner...",✔️ [0.8084081632653062]
4,"Associate Attorney Juan Ramos Law Group, PLLC McAllen, TX Job Details Full-time From $50,000 a year 1 day ago Qualifications Spanish Law Doctoral degree Criminal...","{""position_title"": ""Associate Attorney"", ""location"": ""McAllen, TX"", ""work_arrangement"": ""on-site"", ""experience"": ""None specified."", ""employment_type"": ""full-time"", ""pay"": ""$50,000 a year"", ""degree"": ""Law doctoral degree"", ""certification"": ""Admission to the...","associate attorney juan ramos law group, pllc mcallen, tx job details full-time from $50,000 a year 1 day ago qualifications spanish law doctoral degree criminal...","key-value 

In [62]:
len(llm.history)

16